In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

In [0]:
events = spark.read.table("events_delta")   # or your bronze table

In [0]:
events.display()

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.stream_input
""")

spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.stream_checkpoint
""")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/stream_input"))

In [0]:
events.limit(1000).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/stream_input")

In [0]:
stream_df = spark.readStream \
    .schema(events.schema) \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/stream_input")

In [0]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/stream_checkpoint") \
    .trigger(availableNow=True) \
    .toTable("stream_output_table")

In [0]:
query.status


In [0]:
spark.sql("SELECT COUNT(*) FROM stream_output_table").display()

In [0]:
events.limit(500).write \
    .mode("append") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/stream_input")

In [0]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/stream_checkpoint") \
    .trigger(availableNow=True) \
    .toTable("stream_output_table")

In [0]:
spark.sql("SELECT COUNT(*) FROM stream_output_table").display()